In [10]:
from datetime import datetime

from vnpy_ctastrategy.backtesting import BacktestingEngine
from vnpy_ctastrategy.strategies.atr_rsi_strategy import AtrRsiStrategy
from vnpy_ctastrategy.strategies.boll_channel_strategy import BollChannelStrategy

In [11]:
def run_backtesting(strategy_class, setting, vt_symbol, interval, start, end, rate, slippage, size, pricetick, capital):
    engine = BacktestingEngine()
    engine.set_parameters(
        vt_symbol=vt_symbol,
        interval=interval,
        start=start,
        end=end,
        rate=rate,
        slippage=slippage,
        size=size,
        pricetick=pricetick,
        capital=capital
    )
    engine.add_strategy(strategy_class, setting)
    engine.load_data()
    engine.run_backtesting()
    df = engine.calculate_result()
    return df

def show_portafolio(df):
    engine = BacktestingEngine()
    engine.calculate_statistics(df)
    engine.show_chart(df)

In [12]:
df1 = run_backtesting(
    strategy_class=AtrRsiStrategy,
    setting={},
    vt_symbol="IF88.CFFEX",
    interval="1m",
    start=datetime(2019, 1, 1),
    end=datetime(2019, 4, 30),
    rate=0.3/10000,
    slippage=0.2,
    size=300,
    pricetick=0.2,
    capital=1_000_000,
    )

2025-12-03 16:15:40.882397	开始加载历史数据
2025-12-03 16:15:40.883406	加载进度：# [0%]
2025-12-03 16:15:40.883406	加载进度：# [9%]
2025-12-03 16:15:40.883406	加载进度：## [18%]
2025-12-03 16:15:40.883406	加载进度：### [28%]
2025-12-03 16:15:40.883406	加载进度：#### [37%]
2025-12-03 16:15:40.883406	加载进度：##### [46%]
2025-12-03 16:15:40.883406	加载进度：###### [55%]
2025-12-03 16:15:40.883406	加载进度：####### [65%]
2025-12-03 16:15:40.883406	加载进度：######## [74%]
2025-12-03 16:15:40.883406	加载进度：######### [83%]
2025-12-03 16:15:40.883406	加载进度：########## [92%]
2025-12-03 16:15:40.883406	历史数据加载完成，数据量：0
2025-12-03 16:15:40.884605	策略初始化完成
2025-12-03 16:15:40.884605	开始回放历史数据
2025-12-03 16:15:40.884605	历史数据回放结束
2025-12-03 16:15:40.884605	开始计算逐日盯市盈亏
2025-12-03 16:15:40.884605	回测成交记录为空
2025-12-03 16:15:40.884605	逐日盯市盈亏计算完成


In [13]:
df2 = run_backtesting(
    strategy_class=BollChannelStrategy,
    setting={'fixed_size': 16},
    vt_symbol="RB88.SHFE",
    interval="1m",
    start=datetime(2019, 1, 1),
    end=datetime(2019, 4, 30),
    rate=1/10000,
    slippage=1,
    size=10,
    pricetick=1,
    capital=1_000_000,
    )

2025-12-03 16:15:40.907953	开始加载历史数据
2025-12-03 16:15:40.907953	加载进度：# [0%]
2025-12-03 16:15:40.907953	加载进度：# [9%]
2025-12-03 16:15:40.907953	加载进度：## [18%]
2025-12-03 16:15:40.907953	加载进度：### [28%]
2025-12-03 16:15:40.907953	加载进度：#### [37%]
2025-12-03 16:15:40.907953	加载进度：##### [46%]
2025-12-03 16:15:40.907953	加载进度：###### [55%]
2025-12-03 16:15:40.907953	加载进度：####### [65%]
2025-12-03 16:15:40.907953	加载进度：######## [74%]
2025-12-03 16:15:40.907953	加载进度：######### [83%]
2025-12-03 16:15:40.907953	加载进度：########## [92%]
2025-12-03 16:15:40.907953	历史数据加载完成，数据量：0
2025-12-03 16:15:40.907953	策略初始化完成
2025-12-03 16:15:40.907953	开始回放历史数据
2025-12-03 16:15:40.907953	历史数据回放结束
2025-12-03 16:15:40.907953	开始计算逐日盯市盈亏
2025-12-03 16:15:40.907953	回测成交记录为空
2025-12-03 16:15:40.907953	逐日盯市盈亏计算完成


In [14]:
# 检查两个回测结果是否有效
if df1 is None or df2 is None:
    print("错误: 无法组合回测结果，因为至少有一个回测结果为空")
    print(f"df1 是否为空: {df1 is None}")
    print(f"df2 是否为空: {df2 is None}")
    print("\n提示: 请确保数据库中有历史数据，并且策略产生了成交记录")
else:
    # 确保两个 DataFrame 有相同的索引（日期）
    # 使用外连接合并，然后对数值列求和
    import pandas as pd
    
    # 确保索引是日期类型
    if not isinstance(df1.index, pd.DatetimeIndex):
        if 'date' in df1.columns:
            df1 = df1.set_index('date')
    if not isinstance(df2.index, pd.DatetimeIndex):
        if 'date' in df2.columns:
            df2 = df2.set_index('date')
    
    # 选择需要相加的数值列
    numeric_columns = ['net_pnl', 'total_pnl', 'trading_pnl', 'holding_pnl', 
                      'commission', 'turnover', 'trade_count']
    
    # 只保留两个 DataFrame 都有的列
    common_columns = [col for col in numeric_columns if col in df1.columns and col in df2.columns]
    
    if not common_columns:
        print("错误: 两个 DataFrame 没有共同的数值列可以相加")
        print(f"df1 的列: {df1.columns.tolist()}")
        print(f"df2 的列: {df2.columns.tolist()}")
    else:
        # 合并并相加
        df1_selected = df1[common_columns]
        df2_selected = df2[common_columns]
        
        # 使用外连接合并，然后对数值列求和
        dfp = df1_selected.add(df2_selected, fill_value=0)
        
        # 移除所有值都为 NaN 的行
        dfp = dfp.dropna(how='all')
        
        # 检查结果
        if dfp.empty:
            print("警告: 组合后的 DataFrame 为空")
        else:
            print(f"组合回测结果: {len(dfp)} 条记录")
            print(f"包含列: {dfp.columns.tolist()}")
            show_portafolio(dfp, capital=1_000_000)

错误: 两个 DataFrame 没有共同的数值列可以相加
df1 的列: []
df2 的列: []


In [15]:
# 诊断信息：检查回测结果
print("=" * 60)
print("回测结果诊断")
print("=" * 60)

print(f"\ndf1 类型: {type(df1)}")
if df1 is not None:
    print(f"df1 是否为空: {df1.empty}")
    if not df1.empty:
        print(f"df1 形状: {df1.shape}")
        print(f"df1 列: {df1.columns.tolist()}")
        print(f"df1 索引类型: {type(df1.index)}")
    else:
        print("df1 是空的 DataFrame")
else:
    print("df1 为 None（回测没有产生结果）")

print(f"\ndf2 类型: {type(df2)}")
if df2 is not None:
    print(f"df2 是否为空: {df2.empty}")
    if not df2.empty:
        print(f"df2 形状: {df2.shape}")
        print(f"df2 列: {df2.columns.tolist()}")
        print(f"df2 索引类型: {type(df2.index)}")
    else:
        print("df2 是空的 DataFrame")
else:
    print("df2 为 None（回测没有产生结果）")

print("\n" + "=" * 60)
print("问题诊断:")
print("=" * 60)
if df1 is None or df2 is None or (df1 is not None and df1.empty) or (df2 is not None and df2.empty):
    print("\n⚠ 回测结果为空的原因可能是：")
    print("  1. 数据库中没有历史数据（需要先导入历史数据）")
    print("  2. 策略在回测期间没有产生任何交易信号")
    print("  3. 合约代码不正确（如 'IF88.CFFEX' 可能不存在）")
    print("\n解决方案：")
    print("  1. 使用 DataManager 导入历史数据")
    print("  2. 检查合约代码是否正确")
    print("  3. 调整策略参数或时间范围")
else:
    print("\n✓ 两个回测结果都有效，可以继续组合")
print("=" * 60)


回测结果诊断

df1 类型: <class 'pandas.core.frame.DataFrame'>
df1 是否为空: True
df1 是空的 DataFrame

df2 类型: <class 'pandas.core.frame.DataFrame'>
df2 是否为空: True
df2 是空的 DataFrame

问题诊断:

⚠ 回测结果为空的原因可能是：
  1. 数据库中没有历史数据（需要先导入历史数据）
  2. 策略在回测期间没有产生任何交易信号
  3. 合约代码不正确（如 'IF88.CFFEX' 可能不存在）

解决方案：
  1. 使用 DataManager 导入历史数据
  2. 检查合约代码是否正确
  3. 调整策略参数或时间范围
